# Bagging

Bagging adalah singkatan dari *bootstrap aggregating*. Banyak model dilatih pada sampel acak dengan pengembalian, lalu prediksinya digabung lewat voting. Cara ini menurunkan variansi tanpa menambah bias secara berarti.

## Tujuan

- Memahami bootstrap sampling.
- Membandingkan pohon tunggal dengan bagging.
- Menerapkan bagging pada data churn dan data sel tumor.
- Mencari hyperparameter dengan grid search.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.utils import resample


In [ ]:
def get_accuracy(X_train, X_test, y_train, y_test, model):
    return {
        "test Accuracy": metrics.accuracy_score(y_test, model.predict(X_test)),
        "train Accuracy": metrics.accuracy_score(y_train, model.predict(X_train)),
    }


## Data churn pelanggan

Data pelanggan telekomunikasi dipakai sebagai contoh pertama. Kolom `churn` menjadi target.


In [ ]:
CHURN_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "ChurnData.csv"
)
churn_data = pd.read_csv(CHURN_URL)
churn_data = churn_data[
    ["tenure", "age", "address", "income", "ed", "employ", "equip",
     "callcard", "wireless", "churn"]
]
churn_data["churn"] = churn_data["churn"].astype("int")
churn_data.head()


## Bootstrap sampling

`resample` mengambil baris dengan pengembalian. Perhatikan bahwa sebagian baris bisa muncul dua kali, sementara baris lain hilang.


In [ ]:
churn_data[0:5]


In [ ]:
for _ in range(3):
    print(resample(churn_data[0:5]).index.tolist())


In [ ]:
features = churn_data[["tenure", "age", "address", "income", "ed", "employ", "equip"]]
target = churn_data["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=0
)
print("Train set", X_train.shape, y_train.shape)
print("Test set", X_test.shape, y_test.shape)


## Pohon keputusan tunggal

Pohon tunggal dengan kedalaman terbatas dipakai sebagai pembanding. Pohon dalam mudah overfitting: akurasi latih tinggi, tetapi akurasi uji rendah.


In [ ]:
single_tree = DecisionTreeClassifier(
    criterion="entropy", max_depth=5, random_state=10
).fit(X_train, y_train)
print(get_accuracy(X_train, X_test, y_train, y_test, single_tree))


In [ ]:
figure, axis = plt.subplots(figsize=(18, 8))
plot_tree(single_tree, feature_names=list(features.columns), class_names=["stay", "left"],
          filled=True, fontsize=8, ax=axis)
plt.show()


## Bagging pada data churn

Banyak pohon dangkal dilatih pada sampel bootstrap, lalu prediksinya digabung lewat suara terbanyak.


In [ ]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=2),
    n_estimators=30,
    random_state=0,
    bootstrap=True,
    n_jobs=-1,
).fit(X_train, y_train)
print(get_accuracy(X_train, X_test, y_train, y_test, bagging_model))


## Pengaruh jumlah estimator

Akurasi diuji untuk jumlah estimator 1 sampai 40. Kurva biasanya naik cepat di awal, lalu mendatar.


In [ ]:
scores = []

for n_estimators in range(1, 41):
    model = BaggingClassifier(
        estimator=DecisionTreeClassifier(criterion="entropy", max_depth=10),
        n_estimators=n_estimators,
        bootstrap=True,
        random_state=0,
        n_jobs=-1,
    ).fit(X_train, y_train)
    scores.append(
        {"n_estimators": n_estimators,
         **get_accuracy(X_train, X_test, y_train, y_test, model)}
    )

score_table = pd.DataFrame(scores).set_index("n_estimators")


In [ ]:
axis = score_table.plot(figsize=(10, 6))
axis.set(xlabel="Number of estimators", ylabel="Accuracy")
plt.title("Customer Churn")
plt.tight_layout()
plt.show()


## Bagging dengan model dasar SVM

Bagging tidak hanya berlaku untuk pohon. SVM linear juga bisa dipakai sebagai model dasar.


In [ ]:
svm_model = SVC(kernel="linear", gamma="scale").fit(X_train, y_train)
print("SVM tunggal:", get_accuracy(X_train, X_test, y_train, y_test, svm_model))

bagged_svm = BaggingClassifier(
    estimator=SVC(kernel="linear", gamma="scale"),
    n_estimators=10,
    random_state=0,
    bootstrap=True,
    n_jobs=-1,
).fit(X_train, y_train)
print("SVM dengan bagging:", get_accuracy(X_train, X_test, y_train, y_test, bagged_svm))


## Data sel tumor

Data sampel sel dipakai untuk menguji bagging pada klasifikasi tumor jinak dan ganas, sekaligus mencari hyperparameter terbaik.


In [ ]:
CANCER_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "cell_samples.csv"
)
cell_data = pd.read_csv(CANCER_URL)
cell_data = cell_data[pd.to_numeric(cell_data["BareNuc"], errors="coerce").notnull()]

cell_features = cell_data[
    ["Clump", "UnifSize", "UnifShape", "MargAdh", "SingEpiSize",
     "BareNuc", "BlandChrom", "NormNucl", "Mit"]
].astype("int")
cell_target = cell_data["Class"].astype("int")

X_train, X_test, y_train, y_test = train_test_split(
    cell_features, cell_target, test_size=0.2, random_state=4
)
print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)


In [ ]:
param_grid = {
    "n_estimators": [9, 19, 29],
    "estimator__max_depth": [3, 5, 7],
}

search = GridSearchCV(
    BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=0), random_state=0, n_jobs=-1
    ),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)
search.best_score_


In [ ]:
search.best_params_


In [ ]:
print(get_accuracy(X_train, X_test, y_train, y_test, search.best_estimator_))


## Latihan: data jenis obat

Data resep pasien dipakai untuk menerapkan bagging pada klasifikasi multikelas.


In [ ]:
DRUG_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/"
    "drug200.csv"
)
drug_data = pd.read_csv(DRUG_URL, delimiter=",")

drug_features = drug_data[["Age", "Sex", "BP", "Cholesterol", "Na_to_K"]].copy()
drug_features["Sex"] = drug_features["Sex"].map({"F": 0, "M": 1})
drug_features["BP"] = drug_features["BP"].map({"LOW": 0, "NORMAL": 1, "HIGH": 2})
drug_features["Cholesterol"] = drug_features["Cholesterol"].map(
    {"NORMAL": 0, "HIGH": 1}
)

X_train, X_test, y_train, y_test = train_test_split(
    drug_features, drug_data["Drug"], test_size=0.2, random_state=4
)
X_train.head()


In [ ]:
drug_search = GridSearchCV(
    BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=0), random_state=0, n_jobs=-1
    ),
    param_grid={"n_estimators": [9, 19], "estimator__max_depth": [3, 5]},
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
).fit(X_train, y_train)

drug_search.best_score_, drug_search.best_params_


In [ ]:
print(get_accuracy(X_train, X_test, y_train, y_test, drug_search.best_estimator_))


## Ringkasan

Bagging menurunkan overfitting pada pohon tunggal. Akurasi uji naik cepat seiring jumlah estimator, lalu mendatar setelah puluhan model. Model dasar apa pun bisa dipakai selama mendukung `fit` dan `predict`.
